# NB4 — Medallion Pipeline (Bronze → Silver → Gold), lightweight

**Use case:** LLM observability — exact schema from slide §8 (Lakehouse cho AI/ML) medallion frame.
Maps to deliverable bullet 4 (the Milestone-1 Lakehouse artifact).

Pre-req: ran `make data` — but if you jumped straight here, the cell below
generates the Bronze sample for you rather than failing on a missing path.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path
from pathlib import Path

import polars as pl
import duckdb
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

BRONZE = path("bronze", "llm_calls_raw")
SILVER = path("silver", "llm_calls")
GOLD   = path("gold",   "llm_daily_metrics")

# Self-healing pre-req (same pattern as NB7/NB8). Without this, skipping
# `make data` surfaces as a raw `Os { code: 2, kind: NotFound }` from the Rust
# layer — technically correct, useless to a student.
if not Path(BRONZE).exists():
    print("Bronze not found — running scripts/generate_data_lite.py first ...")
    import generate_data_lite

    generate_data_lite.main()

## Bronze — verify raw is loaded

In [2]:
bronze_n = DeltaTable(BRONZE).to_pyarrow_table().num_rows
print(f"Bronze rows: {bronze_n:,}")
print(pl.from_arrow(DeltaTable(BRONZE).to_pyarrow_table().slice(0, 2)))

Bronze rows: 200,000
shape: (2, 3)
┌─────────────────────────────────┬─────────────────────────┬─────────────────────────────────┐
│ request_id                      ┆ ts                      ┆ raw_json                        │
│ ---                             ┆ ---                     ┆ ---                             │
│ str                             ┆ datetime[μs, UTC]       ┆ str                             │
╞═════════════════════════════════╪═════════════════════════╪═════════════════════════════════╡
│ 795553b5-9d0f-4177-8a00-a8eed7… ┆ 2026-04-01 00:00:00 UTC ┆ {"model": "claude-sonnet-4-6",… │
│ d0152dd3-1037-4632-b41d-9668bc… ┆ 2026-04-01 00:00:03 UTC ┆ {"model": "claude-haiku-4-5", … │
└─────────────────────────────────┴─────────────────────────┴─────────────────────────────────┘


### 📝 Nhận xét — Bronze

- Bronze có **200.000 dòng**, chỉ 3 cột: `request_id`, `ts` (timestamp **UTC**) và `raw_json` — payload log LLM giữ nguyên dạng **chuỗi JSON**, chưa parse, chưa kiểm tra.
- Vai trò của Bronze: **lưu đúng những gì đã nhận**, kể cả bản ghi trùng hay lỗi. Không sửa gì ở tầng này để khi logic làm sạch thay đổi (thêm trường, đổi quy tắc dedup) ta có thể **xử lý lại** Silver/Gold từ nguồn gốc mà không cần ingest lại.
- Bronze là bảng Delta nên vẫn có transaction log, time travel — nhưng chưa có schema nghiệp vụ (mọi thứ nằm trong `raw_json`).


## Silver — parse, validate, dedup

Rules: drop malformed JSON, dedupe by `request_id`, project typed columns.

In [3]:
reset(SILVER)

# DuckDB does the JSON parse + dedup in one query — Polars also works,
# DuckDB just has nicer JSON syntax for this case.
# DuckDB reads Delta through Arrow, not through `delta_scan()`. The latter
# autoloads an extension over the network; Arrow registration is offline and
# zero-copy, so the lab works on a locked-down machine.
con = duckdb.connect()
con.register("bronze", DeltaTable(BRONZE).to_pyarrow_table())

silver_arrow = con.sql(f"""
    WITH parsed AS (
      SELECT
        request_id,
        ts,
        CAST(ts AS DATE)                            AS date,
        json_extract_string(raw_json, '$.model')          AS model,
        json_extract_string(raw_json, '$.user_id')        AS user_id,
        CAST(json_extract(raw_json, '$.usage.input')  AS INTEGER) AS prompt_tokens,
        CAST(json_extract(raw_json, '$.usage.output') AS INTEGER) AS completion_tokens,
        CAST(json_extract(raw_json, '$.latency_ms')   AS INTEGER) AS latency_ms,
        json_extract_string(raw_json, '$.status')         AS status,
        ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ts) AS rn
      FROM bronze
    )
    SELECT request_id, ts, date, model, user_id,
           prompt_tokens, completion_tokens, latency_ms, status
    FROM parsed
    WHERE rn = 1 AND model IS NOT NULL
""").arrow()

write_deltalake(SILVER, silver_arrow, mode="overwrite", partition_by=["date"])

silver_n = DeltaTable(SILVER).to_pyarrow_table().num_rows
print(f"Silver rows: {silver_n:,}  (Bronze {bronze_n:,} → dedup dropped {bronze_n - silver_n:,})")
assert silver_n < bronze_n, (
    "Silver has the same row count as Bronze — dedup did not run. "
    "Did you regenerate Bronze with the latest generator (which injects retries)?"
)

Silver rows: 190,052  (Bronze 200,000 → dedup dropped 9,948)


In [4]:
# Phân tích dedup: Bronze có bao nhiêu request_id bị retry?
print(con.sql("""
    WITH c AS (SELECT request_id, count(*) AS attempts FROM bronze GROUP BY 1)
    SELECT count(*) FILTER (WHERE attempts > 1)                AS dup_request_ids,
           sum(attempts) FILTER (WHERE attempts > 1)           AS rows_in_dup_groups,
           sum(attempts - 1)                                   AS rows_removed_by_dedup,
           max(attempts)                                       AS max_attempts,
           (SELECT count(*) FROM bronze
             WHERE json_extract_string(raw_json, '$.model') IS NULL) AS rows_missing_model
    FROM c
"""))

┌─────────────────┬────────────────────┬───────────────────────┬──────────────┬────────────────────┐
│ dup_request_ids │ rows_in_dup_groups │ rows_removed_by_dedup │ max_attempts │ rows_missing_model │
│      int64      │       int128       │        int128         │    int64     │       int64        │
├─────────────────┼────────────────────┼───────────────────────┼──────────────┼────────────────────┤
│            9687 │              19635 │                  9948 │            4 │                  0 │
└─────────────────┴────────────────────┴───────────────────────┴──────────────┴────────────────────┘



### 📝 Nhận xét — Silver: parse, validate, dedup

- **Parse + ép kiểu:** `json_extract` tách `model`, `user_id`, `usage.input/output`, `latency_ms`, `status` thành cột có kiểu (`INTEGER`, `VARCHAR`); `date` suy ra từ `ts`. Silver là nguồn sự thật **ở mức từng request**, có schema rõ ràng.
- **Dedup:** Bronze có **9.687 `request_id` bị lặp** (19.635 dòng, tối đa **4 lần** cho một request) — mô phỏng **retry** của client. `ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ts)` + `rn = 1` giữ một dòng mỗi request ⇒ loại **9.948 dòng** (= tổng số lần lặp thừa). **Silver 190.052 < Bronze 200.000** ✅.
- **Validate:** `model IS NOT NULL` loại 0 dòng (Bronze không có dòng thiếu `model`) — toàn bộ chênh lệch đến từ dedup.
- **Lựa chọn nghiệp vụ:** `ORDER BY ts` giữ **lần gọi sớm nhất**. Với retry, lần đầu thường là lần thất bại (rate limit, timeout), lần cuối mới là kết quả người dùng nhận. Nếu Gold dùng để đo trải nghiệm thực tế thì nên giữ lần **cuối** (`ORDER BY ts DESC`); nếu đo tải thực tế lên API thì không nên dedup mà đếm mọi lần gọi. Quy tắc dedup phải đi theo câu hỏi mà Gold trả lời.
- Silver được ghi **partition theo `date`** (`date=2026-04-01/`, …) để truy vấn lọc theo ngày bỏ qua cả thư mục.


## Gold — aggregate to (date, model) metrics

In [5]:
reset(GOLD)

# Illustrative cost model — NOT canonical pricing.
# (input USD / 1M tokens, output USD / 1M tokens)
COST_TABLE = """
  VALUES
    ('claude-haiku-4-5',  0.80,  4.00),
    ('claude-sonnet-4-6', 3.00, 15.00),
    ('claude-opus-4-7', 15.00, 75.00)
"""

con.register("silver", DeltaTable(SILVER).to_pyarrow_table())
gold_arrow = con.sql(f"""
    WITH cost(model, c_in, c_out) AS ({COST_TABLE})
    SELECT
      s.date,
      s.model,
      QUANTILE_CONT(s.latency_ms, 0.50) AS p50_latency_ms,
      QUANTILE_CONT(s.latency_ms, 0.95) AS p95_latency_ms,
      SUM(s.prompt_tokens)              AS total_prompt_tokens,
      SUM(s.completion_tokens)          AS total_completion_tokens,
      AVG(CASE WHEN s.status <> 'ok' THEN 1.0 ELSE 0.0 END) AS error_rate,
      (SUM(s.prompt_tokens)     * c.c_in  / 1e6) +
      (SUM(s.completion_tokens) * c.c_out / 1e6) AS cost_usd
    FROM silver s
    JOIN cost c USING (model)
    GROUP BY s.date, s.model, c.c_in, c.c_out
    ORDER BY s.date, s.model
""").arrow()

write_deltalake(GOLD, gold_arrow, mode="overwrite", partition_by=["date"])

# Z-order for fast filter-by-model dashboards
DeltaTable(GOLD).optimize.z_order(["model"])

{'numFilesAdded': 8,
 'numFilesRemoved': 8,
 'filesAdded': '{"avg":2768.75,"max":2770,"min":2766,"totalFiles":8,"totalSize":22150}',
 'filesRemoved': '{"avg":2687.875,"max":2689,"min":2687,"totalFiles":8,"totalSize":21503}',
 'partitionsOptimized': 8,
 'numBatches': 8,
 'totalConsideredFiles': 8,
 'totalFilesSkipped': 0,
 'preserveInsertionOrder': False,
 'plannerStrategy': 'zOrder',
 'preservedStableOrder': False,
 'maxBinSpanFiles': 1}

### 📝 Nhận xét — Gold: tổng hợp theo (date, model)

- `GROUP BY date, model` → mỗi dòng Gold trả lời: *hôm đó, model đó chạy nhanh/chậm thế nào, lỗi bao nhiêu, tốn bao nhiêu tiền?*
  - **`p50_latency_ms` / `p95_latency_ms`** (`QUANTILE_CONT`): p50 là trải nghiệm điển hình, p95 là **đuôi chậm** mà trung bình che mất — ví dụ opus p50 ≈ 3.000 ms nhưng p95 ≈ 6.000 ms. SLO thường đặt trên p95/p99.
  - **`error_rate`** = tỷ lệ `status <> 'ok'` ⇒ tính cả `rate_limited` lẫn `error`.
  - **`cost_usd`** = token × đơn giá / 1M theo **bảng giá minh hoạ của lab** (không phải giá thật), join qua CTE `cost`.
- Gold **partition theo `date`** và **Z-order theo `model`**. Output `optimize` cho thấy 8 file → 8 file (1 file mỗi partition, ~2,7 KB): với Gold chỉ 3 dòng/partition, Z-order gần như không có tác dụng — chỉ mang tính minh hoạ; nó có giá trị khi bảng lớn (NB2).


## Verify Gold

In [6]:
gold_df = pl.from_arrow(DeltaTable(GOLD).to_pyarrow_table())
print(gold_df)

# Slide-5 deliverable: "Gold p50/p95/cost qua ≥ 7 ngày". Make that explicit.
n_dates = gold_df.select("date").n_unique()
n_models = gold_df.select("model").n_unique()
print(
    f"\n──── Gold deliverable metrics ────\n"
    f"  Distinct dates:   {n_dates:>3}   (target ≥ 7)\n"
    f"  Distinct models:  {n_models:>3}\n"
    f"  Total Gold rows:  {gold_df.height:>3}   (= dates × models)"
)
assert n_dates >= 7, (
    f"Gold has only {n_dates} dates — slide deliverable requires ≥ 7. "
    "Re-run `make data` (the generator spreads across 7 UTC days)."
)

shape: (24, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ date       ┆ model      ┆ p50_latenc ┆ p95_laten ┆ total_pro ┆ total_com ┆ error_rat ┆ cost_usd  │
│ ---        ┆ ---        ┆ y_ms       ┆ cy_ms     ┆ mpt_token ┆ pletion_t ┆ e         ┆ ---       │
│ date       ┆ str        ┆ ---        ┆ ---       ┆ s         ┆ okens     ┆ ---       ┆ f64       │
│            ┆            ┆ f64        ┆ f64       ┆ ---       ┆ ---       ┆ f64       ┆           │
│            ┆            ┆            ┆           ┆ decimal[3 ┆ decimal[3 ┆           ┆           │
│            ┆            ┆            ┆           ┆ 8,0]      ┆ 8,0]      ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2026-04-04 ┆ claude-hai ┆ 559.0      ┆ 1125.9    ┆ 15997428  ┆ 7956647   ┆ 0.049224  ┆ 44.62453  │
│            ┆ ku-4-5     ┆            ┆           ┆           ┆           ┆

In [7]:
# Tự kiểm tra các quy tắc Gold mà notebook gốc không assert (CHECKPOINTS §4)
con.register("gold", DeltaTable(GOLD).to_pyarrow_table())
print(con.sql("""
    SELECT count(*)                                              AS n_rows,
           count(DISTINCT date)                                  AS dates,
           count(DISTINCT model)                                 AS models,
           min(k)                                                AS min_models_per_date,
           count(*) FILTER (WHERE p50_latency_ms > p95_latency_ms) AS p50_gt_p95,
           count(*) FILTER (WHERE cost_usd <= 0)                 AS nonpos_cost,
           count(*) FILTER (WHERE error_rate NOT BETWEEN 0 AND 1) AS bad_error_rate,
           round(min(error_rate), 4)                             AS min_error_rate,
           round(max(error_rate), 4)                             AS max_error_rate,
           round(sum(cost_usd), 2)                               AS total_cost_usd
    FROM (SELECT *, count(*) OVER (PARTITION BY date) AS k FROM gold)
"""))
print(con.sql("SELECT status, count(*) AS n FROM silver GROUP BY 1 ORDER BY 2 DESC"))
with pl.Config(tbl_rows=30, tbl_cols=8, fmt_str_lengths=20):
    print(gold_df.sort("date", "model").select(
        "date", "model", "p50_latency_ms", "p95_latency_ms",
        pl.col("error_rate").round(4), pl.col("cost_usd").round(2)))

┌────────┬───────┬────────┬─────────────────────┬────────────┬─────────────┬────────────────┬────────────────┬────────────────┬────────────────┐
│ n_rows │ dates │ models │ min_models_per_date │ p50_gt_p95 │ nonpos_cost │ bad_error_rate │ min_error_rate │ max_error_rate │ total_cost_usd │
│ int64  │ int64 │ int64  │        int64        │   int64    │    int64    │     int64      │     double     │     double     │     double     │
├────────┼───────┼────────┼─────────────────────┼────────────┼─────────────┼────────────────┼────────────────┼────────────────┼────────────────┤
│     24 │     8 │      3 │                   3 │          0 │           0 │              0 │         0.0422 │         0.0618 │        4754.78 │
└────────┴───────┴────────┴─────────────────────┴────────────┴─────────────┴────────────────┴────────────────┴────────────────┴────────────────┘

┌──────────────┬────────┐
│    status    │   n    │
│   varchar    │ int64  │
├──────────────┼────────┤
│ ok           │ 180612 │

### 📝 Nhận xét — Tự kiểm tra Gold (notebook gốc không assert)

| Quy tắc (CHECKPOINTS §4) | Kết quả | |
|---|---|---|
| Bronze / Silver / Gold có trên storage | `_lakehouse/bronze/llm_calls_raw`, `silver/llm_calls`, `gold/llm_daily_metrics` | ✅ |
| Silver < Bronze | 190.052 < 200.000 | ✅ |
| ≥ 7 ngày × 3 model | 8 ngày × 3 model = 24 dòng, mọi ngày đủ 3 model (`min_models_per_date = 3`) | ✅ |
| p50 ≤ p95 | `p50_gt_p95 = 0` | ✅ |
| `cost_usd` > 0 | `nonpos_cost = 0`; tổng ≈ **$4.754,78** | ✅ |
| `error_rate` ∈ [0, 1] | `bad_error_rate = 0`; dao động 0,042 – 0,062 | ✅ |

Đọc số liệu:
- `error_rate` ≈ 5% khớp với Silver: 5.683 `rate_limited` + 3.757 `error` trên 190.052 dòng (bộ sinh dữ liệu: 3% + 2%).
- **Chi phí bị chi phối bởi giá, không phải lưu lượng:** opus chỉ ~10% request nhưng tốn ~$290/ngày, gần bằng sonnet (~$345/ngày, ~60% request), vì đơn giá cao gấp 5 lần. Đây là loại insight FinOps mà Gold sinh ra để trả lời.
- Latency tỉ lệ theo model (haiku p50 ≈ 560 ms, sonnet ≈ 1.390 ms, opus ≈ 3.000 ms) và **ổn định qua các ngày**.
- ⚠️ **Bất thường:** ngày **2026-04-01** và **2026-04-08** có `cost_usd` thấp hẳn (haiku $33 và $13,5 so với ~$46 các ngày khác) trong khi latency/error_rate bình thường → không phải sự cố hệ thống mà là **ngày thiếu dữ liệu** — xem cell bên dưới.


In [8]:
# Minh hoạ (không đổi pipeline): CAST(ts AS DATE) phụ thuộc TimeZone của phiên DuckDB
tz_con = duckdb.connect()
tz_con.register("bronze", DeltaTable(BRONZE).to_pyarrow_table())
for tz in [tz_con.sql("SELECT current_setting('TimeZone')").fetchone()[0], "UTC"]:
    tz_con.sql(f"SET TimeZone = '{tz}'")
    per_day = tz_con.sql("""
        SELECT CAST(ts AS DATE) AS date, count(*) AS n
        FROM bronze GROUP BY 1 ORDER BY 1
    """).fetchall()
    print(f"TimeZone={tz:<14} dates={len(per_day)}  "
          f"first={per_day[0][0]} ({per_day[0][1]:,} rows)  last={per_day[-1][0]} ({per_day[-1][1]:,} rows)")

TimeZone=Asia/Bangkok   dates=8  first=2026-04-01 (20,239 rows)  last=2026-04-08 (8,333 rows)


TimeZone=UTC            dates=7  first=2026-04-01 (28,572 rows)  last=2026-04-07 (28,571 rows)


### 📝 Nhận xét — Vì sao Gold có 8 ngày thay vì 7? (lỗi múi giờ)

- Bronze lưu `ts` theo **UTC**, trải đúng **7 ngày** (01/04 00:00 → 07/04 23:59 UTC). Nhưng `CAST(ts AS DATE)` trên `TIMESTAMP WITH TIME ZONE` dùng **TimeZone của phiên DuckDB** — trên máy chạy notebook này là `Asia/Bangkok` (UTC+7):
  - `Asia/Bangkok` → **8 ngày**: 01/04 chỉ có 20.239 dòng Bronze (từ 07:00), 08/04 chỉ 8.333 dòng (00:00–06:59).
  - `UTC` → **7 ngày** đều nhau, ~28.571 dòng/ngày.
- Hệ quả: hai ngày biên là **ngày thiếu dữ liệu**, nên `cost_usd`/tổng token của chúng thấp giả tạo — dashboard sẽ báo "chi phí giảm mạnh" dù không có gì thay đổi. Kết quả vẫn đạt ngưỡng rubric (8 ≥ 7) nhưng **sai ngữ nghĩa**.
- Quan trọng hơn: **cùng code, cùng dữ liệu, chạy trên server UTC và laptop ở Việt Nam cho ra hai bảng Gold khác nhau** — pipeline không tất định. Partition `date` của Silver cũng bị lệch theo.
- Cách sửa (không áp dụng ở đây để giữ nguyên code đề bài): thêm `con.sql("SET TimeZone = 'UTC'")` ngay sau `duckdb.connect()` trong cell Silver, hoặc chuyển rõ ràng `CAST(timezone('UTC', ts) AS DATE)`. Nguyên tắc: **quy tắc gán ngày phải được khai báo tường minh** (thường là UTC, hoặc múi giờ nghiệp vụ được chọn có chủ đích), không phụ thuộc máy chạy.


## ✅ Deliverable check
- [ ] All three tables exist under `_lakehouse/{bronze,silver,gold}/`
- [ ] Silver has fewer rows than Bronze (dedup worked)
- [ ] Gold spans ≥ 7 dates × 3 models (slide §8 medallion contract)
- [ ] Cost & error_rate columns populated and non-zero

### 🎯 Kết luận NB4

Pipeline medallion biến 200.000 log LLM thô thành 24 dòng metric phục vụ dashboard. **Bronze** giữ nguyên dữ liệu để có thể xử lý lại; **Silver** parse JSON, ép kiểu và khử 9.948 bản ghi retry (190.052 < 200.000) — quy tắc dedup là một quyết định nghiệp vụ; **Gold** tổng hợp p50/p95 latency, error_rate và chi phí theo ngày × model, đạt mọi quy tắc kiểm tra (≥ 7 ngày × 3 model, p50 ≤ p95, cost > 0, error_rate ∈ [0, 1]). Việc tự kiểm tra Gold còn phát hiện một lỗi mà assert không bắt được: ngày được gán theo múi giờ của máy chạy (UTC+7) thay vì UTC, tạo ra 2 ngày thiếu dữ liệu — chất lượng Gold cần được kiểm tra bằng **ngữ nghĩa**, không chỉ bằng ngưỡng số lượng.
